### Importing

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Base Paths
SILVER_BASE_PATH = "abfss://silver@stretaillakehouse01.dfs.core.windows.net/"
GOLD_BASE_PATH   = "abfss://gold@stretaillakehouse01.dfs.core.windows.net/"

# Source Path
SILVER_CUSTOMER_PATH = SILVER_BASE_PATH + "customers/"

# Target Path
GOLD_DIM_CUSTOMER_PATH = GOLD_BASE_PATH + "dim_customer/"

### Reading from silver layer

In [0]:
df_dim_customer = (
    spark.read
         .format("parquet")
         .load(SILVER_CUSTOMER_PATH)
)

In [0]:
df_dim_customer.printSchema()
df_dim_customer.show(5, truncate=False)

root
 |-- Customer_ID: string (nullable = true)
 |-- First_Name: string (nullable = true)
 |-- Last_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Registration_Date: date (nullable = true)
 |-- Customer_Segment: string (nullable = true)

+-----------+----------+---------+----------------------------+-------+---------+-----------------+----------------+
|Customer_ID|First_Name|Last_Name|Email                       |Country|City     |Registration_Date|Customer_Segment|
+-----------+----------+---------+----------------------------+-------+---------+-----------------+----------------+
|C00001     |Chasmum   |Tripathi |chasmum.tripathi@outlook.com|India  |Nashik   |2022-05-09       |Regular         |
|C00002     |Zansi     |Chander  |z.chander@gmail.com         |India  |Bengaluru|2023-03-20       |Regular         |
|C00003     |Samuel    |Johal    |samuel.johal@yahoo.com      |India  |Bhopa

### Row counting 

In [0]:
df_dim_customer.count()

5000

### Null value analysis: We kept the null as we don't have any business requirement but in production environment, it depends upon the requirement

In [0]:
df_dim_customer.select(
    *[
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df_dim_customer.columns
    ]
).show()

+-----------+----------+---------+-----+-------+----+-----------------+----------------+
|Customer_ID|First_Name|Last_Name|Email|Country|City|Registration_Date|Customer_Segment|
+-----------+----------+---------+-----+-------+----+-----------------+----------------+
|          0|         0|        0|   63|      0|   0|                0|               0|
+-----------+----------+---------+-----+-------+----+-----------------+----------------+



In [0]:
df_dim_customer.filter(col("Email").isNull()).show(10, truncate=False)

+-----------+----------+---------+-----+-------+---------+-----------------+----------------+
|Customer_ID|First_Name|Last_Name|Email|Country|City     |Registration_Date|Customer_Segment|
+-----------+----------+---------+-----+-------+---------+-----------------+----------------+
|C00169     |Rushil    |Karnik   |NULL |India  |Bhopal   |2021-09-09       |Regular         |
|C00197     |Pushti    |Parikh   |NULL |India  |Warangal |2025-07-15       |Regular         |
|C00264     |Damini    |Chaudry  |NULL |India  |Hyderabad|2023-10-15       |Regular         |
|C00421     |Aayush    |Balay    |NULL |India  |Vadodara |2023-10-19       |Regular         |
|C00435     |Panini    |Banik    |NULL |India  |Nashik   |2023-12-18       |Regular         |
|C00475     |Eta       |Amble    |NULL |India  |Bhopal   |2022-05-04       |Regular         |
|C00477     |Lekha     |Kakar    |NULL |India  |Bhopal   |2023-05-15       |Regular         |
|C00571     |Bhanumati |Kala     |NULL |India  |Vadodara |20

### Duplicate Detection verification

In [0]:
(df_dim_customer
.groupBy("Customer_ID")
.count()
.filter(col("count")>1).show()
)

+-----------+-----+
|Customer_ID|count|
+-----------+-----+
+-----------+-----+



## Observation
### No duplicate value were found in the dataset

### Writing to gold layer

In [0]:
(df_dim_customer.write
 .mode("overwrite")
 .format("parquet")
 .save(GOLD_DIM_CUSTOMER_PATH)
)

### Reading for gold layer

In [0]:
df_gold_dim_customer = (
    spark.read
    .format("parquet")
    .load(GOLD_DIM_CUSTOMER_PATH)
)


### Final verification

In [0]:
df_gold_dim_customer.printSchema()

df_gold_dim_customer.show(10, truncate=False)

print(f"Gold Row Count: {df_gold_dim_customer.count()}")

root
 |-- Customer_ID: string (nullable = true)
 |-- First_Name: string (nullable = true)
 |-- Last_Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Registration_Date: date (nullable = true)
 |-- Customer_Segment: string (nullable = true)

+-----------+----------+---------+----------------------------+-------+---------+-----------------+----------------+
|Customer_ID|First_Name|Last_Name|Email                       |Country|City     |Registration_Date|Customer_Segment|
+-----------+----------+---------+----------------------------+-------+---------+-----------------+----------------+
|C00001     |Chasmum   |Tripathi |chasmum.tripathi@outlook.com|India  |Nashik   |2022-05-09       |Regular         |
|C00002     |Zansi     |Chander  |z.chander@gmail.com         |India  |Bengaluru|2023-03-20       |Regular         |
|C00003     |Samuel    |Johal    |samuel.johal@yahoo.com      |India  |Bhopa